In [3]:
from pathlib import Path
import json
import pandas as pd

from src.verifier.remove_random_visible_objects import remove_random_visible_objects
from src.verifier.rotate_random_visible_objects import rotate_random_visible_objects
from src.verifier.scale_random_visible_objects import scale_random_visible_objects
from src.verifier.translate_random_visible_objects import translate_random_visible_objects

from src.vlm_score import compute_vlm_score_per_object
from src.front3d.render_dataset import render_dataset


## Create Perturbation Datasets

In [ ]:
# for version in ["a", "b"]:
#     target_dir = Path(f"../perturbation_dataset_{version}")
#     target_dir.mkdir(parents=True, exist_ok=True)

#     for scene_dir in Path("/home/jonathansickert/git/3DFrontBench/dataset").iterdir():
#         if scene_dir.name.startswith("."):
#             continue
#         remove_random_visible_objects(scene_dir, target_dir / f"remove/{scene_dir.name}")
#         rotate_random_visible_objects(scene_dir, target_dir / f"rotate/{scene_dir.name}")
#         scale_random_visible_objects(scene_dir, target_dir / f"scale/{scene_dir.name}")
#         translate_random_visible_objects(scene_dir, target_dir / f"translate/{scene_dir.name}")

## Render Perturbation Datasets

In [ ]:
# for version in ["a", "b"]:
#    for subset in ["remove", "rotate", "scale", "translate"]:
#         dataset_dir = Path(f"../perturbation_dataset_{version}/{subset}")
#         render_dataset(dataset_dir=dataset_dir)

In [1]:
model_choice = "gemini"
resize_to_hd = True
run_id = 0

In [ ]:
for version in ["a", "b"]:
    scores = {}

    for scene_dir in Path(f"/home/jonathansickert/git/3DFrontBench/perturbation_dataset_{version}/remove").iterdir():
        if scene_dir.name.startswith("."):
            continue

        score = compute_vlm_score_per_object(
            target_path=Path("/home/jonathansickert/git/3DFrontBench/dataset") / scene_dir.name / "color.png",
            render_path=scene_dir / "color.png",
            score_type="count",
            model_choice=model_choice,
            resize_to_hd=resize_to_hd
        )

        scores[scene_dir.name] = score
        print(score)

    with open(f"count_scores_{model_choice}_{run_id}_{version}.json", "w") as f:
        json.dump(scores, f, indent=2)

In [19]:
for version in ["a", "b"]:
    scores = {}

    for scene_dir in Path(f"/home/jonathansickert/git/3DFrontBench/perturbation_dataset_{version}/translate").iterdir():
        if scene_dir.name.startswith("."):
            continue

        score = compute_vlm_score_per_object(
            target_path=Path("/home/jonathansickert/git/3DFrontBench/dataset") / scene_dir.name / "color.png",
            render_path=scene_dir / "color.png",
            score_type="placement",
            model_choice=model_choice,
            resize_to_hd=resize_to_hd
        )

        scores[scene_dir.name] = score

    with open(f"placement_scores_{model_choice}_{run_id}_{version}.json", "w") as f:
        json.dump(scores, f, indent=2)

In [5]:
for version in ["a", "b"]:
    scores = {}

    for scene_dir in Path(f"/home/jonathansickert/git/3DFrontBench/perturbation_dataset_{version}/scale").iterdir():
        if scene_dir.name.startswith("."):
            continue

        score = compute_vlm_score_per_object(
            target_path=Path("/home/jonathansickert/git/3DFrontBench/dataset") / scene_dir.name / "color.png",
            render_path=scene_dir / "color.png",
            score_type="scale",
            model_choice=model_choice,
            resize_to_hd=resize_to_hd,
        )

        scores[scene_dir.name] = score

    with open(f"scale_scores_{model_choice}_{run_id}_{version}.json", "w") as f:
        json.dump(scores, f, indent=2)

In [4]:
for version in ["a", "b"]:
    scores = {}

    for scene_dir in Path(f"/home/jonathansickert/git/3DFrontBench/perturbation_dataset_{version}/rotate").iterdir():
        if scene_dir.name.startswith("."):
            continue

        score = compute_vlm_score_per_object(
            target_path=Path("/home/jonathansickert/git/3DFrontBench/dataset") / scene_dir.name / "color.png",
            render_path=scene_dir / "color.png",
            score_type="rotation",
            model_choice=model_choice,
            resize_to_hd=resize_to_hd,
        )

        scores[scene_dir.name] = score

    with open(f"rotate_scores_{model_choice}_{run_id}_{version}.json", "w") as f:
        json.dump(scores, f, indent=2)

## Evaluation

Discover every `{experiment}_scores_{model}_{run_id}_{version}.json` file produced above (there can be several per experiment, one per model/run), join each with the ground-truth perturbation score, and summarize AB agreement + Spearman rank correlation into a single dataframe (one row per experiment x model x run).

In [6]:
import re

from scipy.stats import spearmanr

PERTURBATION_ROOT = Path("/home/jonathansickert/git/3DFrontBench")

EXPERIMENT_TO_PERTURBATION_SUBSET = {
    "count": "remove",
    "placement": "translate",
    "scale": "scale",
    "rotate": "rotate",
}

SCORE_FILE_PATTERN = re.compile(
    r"^(?P<experiment>count|placement|scale|rotate)_scores_(?P<model>[^_]+)_(?P<run_id>\d+)_(?P<version>[ab])\.json$"
)


def discover_score_files(score_dir: Path) -> pd.DataFrame:
    """Find all `{experiment}_scores_{model}_{run_id}_{version}.json` files in `score_dir`."""
    rows = []
    for path in sorted(score_dir.glob("*_scores_*.json")):
        match = SCORE_FILE_PATTERN.match(path.name)
        if match:
            rows.append({**match.groupdict(), "path": path})

    df = pd.DataFrame(rows)
    if not df.empty:
        df["run_id"] = df["run_id"].astype(int)
    return df

In [7]:
def build_ab_df(path_a: Path, path_b: Path, experiment: str) -> pd.DataFrame:
    """Join the VLM scores in `path_a`/`path_b` with the ground-truth perturbation score for each scene."""
    dataset_name = EXPERIMENT_TO_PERTURBATION_SUBSET[experiment]

    with open(path_a) as f:
        vlm_score_a = json.load(f)
    with open(path_b) as f:
        vlm_score_b = json.load(f)

    for version, vlm_score in [("a", vlm_score_a), ("b", vlm_score_b)]:
        subset_dir = PERTURBATION_ROOT / f"perturbation_dataset_{version}" / dataset_name
        for scene_dir in subset_dir.iterdir():
            if scene_dir.name.startswith(".") or scene_dir.name not in vlm_score:
                continue
            with open(scene_dir / "score.json") as f:
                vlm_score[scene_dir.name]["score_perturbation"] = json.load(f)["score"]

    rows = []
    for scene in sorted(set(vlm_score_a) & set(vlm_score_b)):
        entry_a, entry_b = vlm_score_a[scene], vlm_score_b[scene]
        if "score_perturbation" not in entry_a or "score_perturbation" not in entry_b:
            continue

        vlm_a, vlm_b = entry_a["score"], entry_b["score"]
        perturbation_a = 1 - entry_a["score_perturbation"]
        perturbation_b = 1 - entry_b["score_perturbation"]

        rows.append(
            {
                "scene": scene,
                "vlm_a": vlm_a,
                "vlm_b": vlm_b,
                "perturbation_a": perturbation_a,
                "perturbation_b": perturbation_b,
                "match": (vlm_a > vlm_b) == (perturbation_a > perturbation_b),
            }
        )

    return pd.DataFrame(rows).set_index("scene")

In [10]:
def summarize_group(ab_df: pd.DataFrame) -> pd.Series:
    """AB agreement rate and Spearman rank correlation (VLM score vs. perturbation score) for one group."""
    vlm_stacked = pd.concat([ab_df["vlm_a"], ab_df["vlm_b"]], ignore_index=True)
    perturbation_stacked = pd.concat([ab_df["perturbation_a"], ab_df["perturbation_b"]], ignore_index=True)
    rho, pvalue = spearmanr(vlm_stacked, perturbation_stacked)

    return pd.Series(
        {
            "n_scenes": len(ab_df),
            "ab_agreement": ab_df["match"].mean(),
            "spearman_rho": rho,
            # "spearman_pvalue": pvalue,
        }
    )

In [11]:
score_files = discover_score_files(Path("."))

ab_dfs = {}
summary_rows = []

for (experiment, model, run_id), group in score_files.groupby(["experiment", "model", "run_id"]):
    versions = dict(zip(group["version"], group["path"]))
    if not {"a", "b"} <= versions.keys():
        print(f"skipping {experiment}/{model}/run {run_id}: missing version(s) {set('ab') - versions.keys()}")
        continue

    ab_df = build_ab_df(versions["a"], versions["b"], experiment)
    ab_dfs[(experiment, model, run_id)] = ab_df

    row = {"experiment": experiment, "model": model, "run_id": run_id}
    row.update(summarize_group(ab_df))
    summary_rows.append(row)

results_df = (
    pd.DataFrame(summary_rows)
    .sort_values(["experiment", "model", "run_id"])
    .reset_index(drop=True)
)
results_df

,experiment,model,run_id,n_scenes,ab_agreement,spearman_rho
0,count,gemini,0,30.0,0.900000,0.958558
1,count,qwen,0,30.0,0.966667,0.924248
2,placement,gemini,0,30.0,0.566667,0.545638
3,placement,qwen,0,30.0,0.600000,0.381556
4,rotate,gemini,0,30.0,0.466667,0.159293
5,rotate,qwen,0,30.0,0.566667,0.032258
6,scale,gemini,0,30.0,0.700000,0.510360
7,scale,qwen,0,30.0,0.566667,-0.085898
